# Tables: read them, change them

Two ways to work with the project's tables from a notebook, and when to use each:

| | **SQL** (`buckets_lake.sql`) | **Spark** (`buckets_lake.spark`) |
|---|---|---|
| Runs | in Trino, as you | on Spark executors started for you |
| Sees | every table you may read, under your row filters and masks | only tables you read **whole** (no filter or mask applies to you) |
| Brings back | at most 100,000 rows, as pandas | any size: the work stays on the executors |
| Changes | `INSERT`, `UPDATE`, `DELETE`, `MERGE`, `CREATE TABLE` where you're an editor | `append`, `MERGE`, `UPDATE`, `DELETE` on tables you may change; new tables via `create=` |
| Needs | any notebook | the **Distributed Spark** profile |

**To use it:** save a copy (File → Save Notebook As…), change the parameters below, and run top to bottom. The examples are the guide's `retail` project (`uploads_orders`); the notebook makes its own table, `customer_tiers`, and changes only that. Changing tables needs `write-tables` in the project (an editor).

In [ ]:
import os

PROJECT = os.environ.get("STORSCALE_PROJECT", "retail")   # the project whose tables these are
SCHEMA = PROJECT.replace("-", "_")   # its namespace, as Spark names tables (SCHEMA.table); Trino's sql() needs only the table's name
ORDERS = "uploads_orders"            # a table to read: order_id, customer_id, ordered_on, region, amount, refunded, test
TIERS = "customer_tiers"             # the table this notebook makes and changes
SUMMARY = "customer_tier_summary"    # a table the Spark part makes

## Part 1: SQL, through Trino

`buckets_lake.sql(project, statement)` runs one statement in Trino as you, with the project's own table names (`FROM uploads_orders`, not `iceberg.retail.uploads_orders`). Ranger's rules apply: a row filter or mask on you applies here too. An answer has at most 100,000 rows, so aggregate in SQL and bring back the result.

### Read

In [ ]:
import buckets_lake

by_region = buckets_lake.sql(PROJECT, f"""
SELECT region, count(*) AS orders, CAST(sum(amount) AS double) AS revenue
FROM {ORDERS}
WHERE NOT test AND NOT refunded
GROUP BY region
ORDER BY revenue DESC
""")
by_region

### Make a table

`CREATE TABLE IF NOT EXISTS ... AS SELECT ... WITH NO DATA` makes an empty table whose columns take their types from the query: `customer_id` is whatever type it is in `uploads_orders`. Run it as often as you like; it makes the table once.

In [ ]:
buckets_lake.sql(PROJECT, f"""
CREATE TABLE IF NOT EXISTS {TIERS} AS
SELECT customer_id,
       CAST(NULL AS varchar) AS tier,
       CAST(0 AS double) AS revenue,
       current_timestamp AS updated_at
FROM {ORDERS}
WITH NO DATA
""")

### Insert

Rows from a query. A statement that changes a table answers with one row: how many rows it changed.

In [ ]:
buckets_lake.sql(PROJECT, f"DELETE FROM {TIERS}")   # start from empty, so the notebook can be run again

buckets_lake.sql(PROJECT, f"""
INSERT INTO {TIERS}
SELECT customer_id,
       CASE WHEN sum(amount) >= 500 THEN 'gold' WHEN sum(amount) >= 200 THEN 'silver' ELSE 'bronze' END,
       CAST(sum(amount) AS double),
       current_timestamp
FROM {ORDERS}
WHERE NOT test AND NOT refunded AND ordered_on < DATE '2026-09-01'
GROUP BY customer_id
""")
after_insert = buckets_lake.sql(PROJECT, f'''SELECT snapshot_id FROM "{TIERS}$snapshots" ORDER BY committed_at DESC LIMIT 1''').snapshot_id[0]
buckets_lake.sql(PROJECT, f"SELECT tier, count(*) AS customers FROM {TIERS} GROUP BY tier ORDER BY tier")

### Update and delete

In [ ]:
buckets_lake.sql(PROJECT, f"UPDATE {TIERS} SET tier = 'platinum', updated_at = current_timestamp WHERE revenue >= 1000")
buckets_lake.sql(PROJECT, f"DELETE FROM {TIERS} WHERE revenue < 20")
buckets_lake.sql(PROJECT, f"SELECT tier, count(*) AS customers, min(revenue) AS least FROM {TIERS} GROUP BY tier ORDER BY least")

### Merge: update the rows that are there, insert the rest

`MERGE` brings the table up to date from a query in one change: here, September's orders added to each customer's revenue, and customers new in September added.

In [ ]:
buckets_lake.sql(PROJECT, f"""
MERGE INTO {TIERS} t
USING (
    SELECT customer_id, CAST(sum(amount) AS double) AS revenue
    FROM {ORDERS}
    WHERE NOT test AND NOT refunded AND ordered_on >= DATE '2026-09-01'
    GROUP BY customer_id
) s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET revenue = t.revenue + s.revenue, updated_at = current_timestamp
WHEN NOT MATCHED THEN INSERT (customer_id, tier, revenue, updated_at)
                      VALUES (s.customer_id, 'new', s.revenue, current_timestamp)
""")

### A few rows from pandas

For a handful of rows (corrections, a lookup list), write them into the statement as `VALUES`. For many rows, use Spark (Part 2), or put a file in the project's landing area and import it.

In [ ]:
import numbers

import pandas as pd


def values(df):
    """A small DataFrame as SQL VALUES rows: text quoted, None as NULL."""
    def lit(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return "NULL"
        if isinstance(v, bool):
            return "TRUE" if v else "FALSE"
        if isinstance(v, numbers.Number):
            return str(v)
        return "'" + str(v).replace("'", "''") + "'"
    return ",\n".join("(" + ", ".join(lit(v) for v in row) + ")" for row in df.itertuples(index=False))


ids = buckets_lake.sql(PROJECT, f"SELECT customer_id FROM {TIERS} ORDER BY revenue DESC LIMIT 2")["customer_id"]
fixes = pd.DataFrame({"customer_id": ids.tolist(), "tier": ["key account", "key account"]})

buckets_lake.sql(PROJECT, f"""
MERGE INTO {TIERS} t
USING (VALUES {values(fixes)}) AS s (customer_id, tier)
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET tier = s.tier, updated_at = current_timestamp
""")
buckets_lake.sql(PROJECT, f"SELECT * FROM {TIERS} ORDER BY revenue DESC LIMIT 5")

### Its history

Each change is a snapshot of the table (Iceberg). `"<table>$snapshots"` lists them, and `FOR VERSION AS OF` reads the table as it was at one. Older snapshots are cleared by the platform's maintenance after a while.

In [ ]:
from IPython.display import display

snapshots = buckets_lake.sql(PROJECT, f'''SELECT snapshot_id, committed_at, operation FROM "{TIERS}$snapshots" ORDER BY committed_at''')
display(snapshots)
# The table as it was just after the INSERT (after_insert, kept then), beside it now:
then = buckets_lake.sql(PROJECT, f"SELECT tier, count(*) AS customers FROM {TIERS} FOR VERSION AS OF {after_insert} GROUP BY tier")
now = buckets_lake.sql(PROJECT, f"SELECT tier, count(*) AS customers FROM {TIERS} GROUP BY tier")
then.merge(now, on="tier", how="outer", suffixes=("_then", "_now")).fillna(0)

## Part 2: Spark, for work too big to bring back

Start the notebook with the **Distributed Spark** profile (File → Hub Control Panel → Stop My Server, then Start My Server). `buckets_lake.spark()` gives a session on the tables you read **whole**; tables with a row filter or mask for you stay Trino's (`spark_skipped` says why). Spark names tables `SCHEMA.table`.

`create=` has the platform make a table first, as you, where you may make tables: Spark's own `CREATE TABLE`, `DROP TABLE` and `ALTER TABLE ... RENAME` are refused. Name the tables the session needs with `tables=`: it takes about thirty at most.

In [ ]:
spark = buckets_lake.spark(
    tables=[f"{SCHEMA}.{ORDERS}", f"{SCHEMA}.{TIERS}"],   # and the ones create= makes
    create={f"{SCHEMA}.{SUMMARY}": {"tier": "varchar", "customers": "bigint", "revenue": "double"}},
)
print("tables:", buckets_lake.spark_tables)     # name -> "read" or "read and write"
print("skipped:", buckets_lake.spark_skipped)   # name -> why

### Read

In [ ]:
from pyspark.sql import functions as F

orders = spark.table(f"{SCHEMA}.{ORDERS}").where(~F.col("test") & ~F.col("refunded"))
orders.groupBy("region").agg(F.count("*").alias("orders"), F.sum("amount").cast("double").alias("revenue")).orderBy("region").show()

tiers = spark.table(f"{SCHEMA}.{TIERS}")
tiers.orderBy(F.desc("revenue")).show(5)

### Write a whole table

`writeTo(...).append()` adds rows; `overwritePartitions()` replaces what's there (the whole table, when it isn't partitioned).

In [ ]:
summary = (tiers.groupBy("tier")
                .agg(F.count("*").alias("customers"), F.sum("revenue").alias("revenue")))
summary.writeTo(f"{SCHEMA}.{SUMMARY}").overwritePartitions()
spark.table(f"{SCHEMA}.{SUMMARY}").orderBy("revenue").show()

### Merge, update and delete

Spark SQL takes Iceberg's `MERGE INTO`, `UPDATE` and `DELETE FROM`. A DataFrame becomes a merge's source as a temporary view.

In [ ]:
latest = (orders.groupBy("customer_id")
                .agg(F.sum("amount").cast("double").alias("revenue"))
                .withColumn("tier", F.when(F.col("revenue") >= 500, "gold").when(F.col("revenue") >= 200, "silver").otherwise("bronze"))
                .withColumn("updated_at", F.current_timestamp())
                .select("customer_id", "tier", "revenue", "updated_at"))   # the table's columns, in its order
latest.createOrReplaceTempView("latest")

spark.sql(f"""
MERGE INTO {SCHEMA}.{TIERS} t
USING latest s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t.tier <> 'key account' THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")
spark.sql(f"UPDATE {SCHEMA}.{TIERS} SET tier = 'gold' WHERE tier = 'silver' AND revenue >= 450")
spark.sql(f"DELETE FROM {SCHEMA}.{TIERS} WHERE revenue < 20")
spark.table(f"{SCHEMA}.{TIERS}").groupBy("tier").count().orderBy("tier").show()

## Tidy up

The two tables are the notebook's own. To drop them, uncomment and run: dropping is Trino's, as you.

In [ ]:
# for t in (TIERS, SUMMARY):
#     buckets_lake.sql(PROJECT, f"DROP TABLE IF EXISTS {t}")